In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier

# Load data
tr = pd.read_csv("/kaggle/input/competitions/re-ca-2/train.csv")
te = pd.read_csv("/kaggle/input/competitions/re-ca-2/test.csv")

X, y = tr.drop(columns=["id","label"]), tr.label
Xt, ids = te[X.columns], te.id

# Select model (uncomment one)
# model = LogisticRegression(max_iter=1000)
# model = KNeighborsClassifier()
# model = DecisionTreeClassifier()
# model = RandomForestClassifier()
# model = SVC(probability=True)
model = CatBoostClassifier(verbose=0)

# Preprocessing
X = pd.get_dummies(X)
Xt = pd.get_dummies(Xt).reindex(columns=X.columns, fill_value=0)

# Split and train
a,b,ya,yb = train_test_split(X,y,test_size=0.2,random_state=42)

if not isinstance(model, CatBoostClassifier):
    model = make_pipeline(SimpleImputer(), StandardScaler(), model)

model.fit(a,ya)

# Evaluation
p = model.predict(b)
print("Accuracy:", accuracy_score(yb,p))
print("Precision:", precision_score(yb,p,zero_division=0))
print("Recall:", recall_score(yb,p,zero_division=0))
print("F1:", f1_score(yb,p,zero_division=0))
print("Confusion Matrix:\n",confusion_matrix(yb,p))

# Final training and submission
model.fit(X,y)
sub = pd.DataFrame({"id":ids,"label":model.predict(Xt)})
sub.to_csv("/kaggle/working/submission.csv",index=False)
print(sub.head())


In [ ]:

import glob, os, pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import *

# Load data
tr = pd.read_csv("/kaggle/input/competitions/re-ca-2/train.csv")
te = pd.read_csv("/kaggle/input/competitions/re-ca-2/test.csv")

# Prepare data
X = tr.drop(columns=["id","label"]).copy()
y = tr["label"]
Xt = te[X.columns].copy()
ids = te["id"]

cat = X.select_dtypes("object").columns.tolist()
for c in cat:
    X[c] = X[c].fillna("Missing").astype(str)
    Xt[c] = Xt[c].fillna("Missing").astype(str)

# Split data
a,b,ya,yb = train_test_split(X,y,test_size=.2,random_state=42)

# Train model
m = CatBoostClassifier(verbose=0)
m.fit(a,ya,cat_features=cat,eval_set=(b,yb),early_stopping_rounds=100)

# Evaluate
prob = m.predict_proba(b)[:,1]
pred = (prob>=.5).astype(int)

print("Accuracy:",accuracy_score(yb,pred))
print("Precision:",precision_score(yb,pred,zero_division=0))
print("Recall:",recall_score(yb,pred,zero_division=0))
print("F1 Score:",f1_score(yb,pred))
print("ROC-AUC:",roc_auc_score(yb,prob))
print("Confusion Matrix:\n",confusion_matrix(yb,pred))

# Feature importance
print("\nFeature Importance:\n",
      pd.Series(m.feature_importances_,index=X.columns).sort_values(ascending=False))

# Final training
m.fit(X,y,cat_features=cat)

# Create submission
sub = pd.DataFrame({
    "id":ids,
    "label":m.predict(Xt)
})

sub.to_csv("/kaggle/working/submission.csv",index=False)
print("\nSubmission saved successfully!")
print(sub.head())


In [ ]:
from IPython.display import display, HTML

display(HTML("""
<a href="https://chatgpt.com" target="_blank">🤖</a>
<a href="https://gemini.google.com" target="_blank">✨</a>
"""))